In [2]:
from importlib import reload

## Signature1000 config keys

In [85]:
keys = dict(
    getplan = ['MIAVG','AVG','DIAVG','VD','MV','SA','BURST','MIBURST',
               'DIBURST','SV','FN','SO','FREQ','NSTT'],
    getavg = ['NC','CS','BD','CY','PL','AI','VP','VR','DF','NPING','NB',
              'CH','MUX','BW','ALTI','BT','ICE','ALTISTART','ALTIEND',
              'RAWALTI'],
    gettmavg = ['EN','CD','PD','AVG','TV','TA','TC','CY','FO','SO','DF','DISTILT','TPG','MAPBINS','CORRTH'],
    
    getinst = ['BR','RS','LED','ORIENT','CMTOUT','DMTOUT','CFMTOUT'],
    
    getburst = ['NC','NB','CS','BD','CY','PL','SR','NS','VR','VP','DF',
                'NPING','CH','ALTI','VR5','BT','DISV','ECHO','RAWALTI',
                'ALTISTART','ALTIEND','HR','HR5'],
    
    getmem = ['PLAN','BURST','AVG','PLAN1','BURST1','AVG1','TOTAL'],
    
    getpwr = ['PLAN','BURST','AVG','PLAN1','BURST1','AVG1','TOTAL'],
)

def format_response(command,response):
    command = command.decode('ascii').strip().lower()
    if command in keys.keys():
        response_keys = keys[command]
        formatted = response.partition('\n')[0].strip().split(',')
        return dict(zip(response_keys, formatted))
    else:
        return response


# Serial connection

### Connect to instrument

In [3]:
import serial
from importlib import reload

ser = serial.Serial(
    port='COM4',
    baudrate=115200,
    timeout=1
)

### Stopping failsafe

In [8]:
record_data = False
stop_thread = True
reader_thread.join()
ser.close()

In [9]:
ser.close()

### Read output

In [ ]:
import threading
import time
from source import serial as serial_tools
reload(serial_tools)

# Stop any existing thread
try:
    if reader_thread.is_alive():
        stop_thread = True
        reader_thread.join()
        time.sleep(0.5)
except NameError:
    pass

# Control flags
stop_thread = False
recordtm_data = False
clean_data = True

# Reader function
def read_from_port(ser):
    global stop_thread, record_data
    buffer = b''
    while not stop_thread:
        try:
            data = ser.read(ser.in_waiting or 1)
            if data:
                buffer += data
                while b'\n' in buffer:
                    line, buffer = buffer.split(b'\n', 1)
                    decoded = line.decode('utf-8', errors='ignore') + '\n'
                    print(decoded, end='')
                    if record_data:
                        serial_tools.record_data_to_file(decoded, savepath, clean_data)
        except Exception as e:
            print("Read error:", e)
            stop_thread = True

# Start thread
reader_thread = threading.Thread(
    target=read_from_port,
    args=(ser,),
    daemon=True
)
reader_thread.start()


### Write commands

In [7]:
from source import serial as serial_tools
import os

# Control flags
stop_thread = False
record_data = False
clean_data = True

# Run input loop
try:
    print("Type commands (type 'exit' to exit input mode, 'disconnect' to disconnect):")
    while True:
        # Receive command
        cmd = input(">>> ")

        # Interpret command
        if cmd.lower() == 'disconnect':
            stop_thread = True
            reader_thread.join()
            ser.close()
            print(' Connection terminated.')
            break
        elif cmd.lower() == 'exit':
            print(' Input closed. Connection idle.')
            break
        elif cmd.lower() == 'record_data':
            record_data = not record_data
            print(f'record_data = {record_data}')
        elif cmd.lower() == 'clean_data':
            clean_data = not clean_data
            print(f'clean_data = {clean_data}')
        elif cmd.lower() == 'break':
            serial_tools.send_break(ser)
        elif cmd.lower() == 'downloadtm':
            # Set file to be recorded
            record_data = True
            encoded_cmd = f'{cmd}\r'.encode('utf-8')
            ser.write(encoded_cmd)
            print(' Downloading...')
            # Close input
            break
        else:
            encoded_cmd = f'{cmd}\r'.encode('utf-8')
            ser.write(encoded_cmd)

except Exception as e:
    print("Error:", e)


Type commands (type 'exit' to exit input mode, 'disconnect' to disconnect):


>>>  break
>>>  id
>>>  listfiles,"la"
>>>  download,fn="c03_dry_avgd.ad2cp"
>>>  break
>>>  record_data


record_data = True


>>>  download,fn="c03_dry_avgd.ad2cp"
>>>  record_data


record_data = False


...

record_data = True


>>>  download,fn="c03_dry_avgd.ad2cp"
>>>  record_data


record_data = False


>>>  disconnect


 Connection terminated.


In [6]:
savepath = 'recorded_data/c03_ad2cp_test.txt'

## Verify data integrity

In [36]:
import numpy as np

# Checks if lines match with their checksums
def verify_checksum(lines):
    verified = []
    for line in lines:
        if not line.startswith('$') or '*' not in line:
            verified.append(False)
            continue
        content, checksum_str = line[1:].split('*', 1)
        computed = 0
        for char in content:
            computed ^= ord(char)
        verified.append(f"{computed:02X}" == checksum_str.upper())    
    return np.array(verified)

# Read in recorded telemetry data
with open(savepath, 'r') as file:
    lines = file.readlines()
    lines = np.array([line.strip() for line in lines if len(line) > 1])


### Download new telemetry file

In [69]:
# Set file to be recorded
record_data = True
# Delete existing file
if os.path.exists(savepath):
    os.remove(savepath)
# Download file
ser.write(b'downloadtm\r')
print('Downloading...')

Downloading...


### Check integrity

In [70]:
# Check data integrity
checks = verify_checksum(lines)

print(sum(checks),'/',len(checks))
print(all(checks))

2200 / 2200
True


### Inspect broken lines

In [44]:
ind = np.where(~checks)

for line in lines[ind]:
    print(line, end='\n\n')